# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² dataset using the `mlcroissant` library. The dataset includes rich clinicopathological and molecular information about secondary primary colorectal cancers in cancer survivors.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant dataset schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset title: {metadata.name}")
print(f"Description: {metadata.description}")

## 2. Data Overview
Review available record sets and fields, referencing all entities by their `@id`.

We'll list all record sets, their fields, and columns using their `@id`s.

In [ ]:
# List all record sets by @id
record_sets = list(dataset.record_sets)
print("Available Record Set @id values:")
for rs in record_sets:
    print(f"- {rs['@id']} : {rs['name'] if 'name' in rs else ''}")
    print("  Fields by @id:")
    for field in rs.get('field', []):
        if isinstance(field, dict):
            # Expanded field
            print(f"    - {field['@id']} : {field.get('name','')}")
        else:
            # field is an @id string
            print(f"    - {field}")
    if 'column' in rs:
        print("  Columns by @id:")
        for col in rs['column']:
            if isinstance(col, dict):
                print(f"    - {col['@id']} : {col.get('name','')}")
            else:
                print(f"    - {col}")
    print()

## 3. Data Extraction
Load data from the record sets into pandas DataFrames for analysis.
We will extract data using the `@id` of the record set and use field or column `@id`s as DataFrame columns.

In [ ]:
# List of record set @ids
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded {len(dataframes[record_set_id])} records for {record_set_id}.")
    except Exception as e:
        print(f"Could not load records for {record_set_id}: {e}")
        dataframes[record_set_id] = None
print("")
# Show columns of first available record set
for rs_id, df in dataframes.items():
    if isinstance(df, pd.DataFrame) and not df.empty:
        print(f"Fields/Columns in record set {rs_id}:")
        print(df.columns.tolist())
        display(df.head())
        break

## 4. Exploratory Data Analysis (EDA)
We'll perform EDA on one of the record sets. We'll select a numeric field (by its `@id`) for processing, e.g., to filter, normalize, and group data.

In [ ]:
# Find the first DataFrame with numeric fields
import numpy as np
# Select first non-empty DataFrame
selected_rs_id = None
df = None
for rs_id, frame in dataframes.items():
    if isinstance(frame, pd.DataFrame) and not frame.empty:
        df = frame
        selected_rs_id = rs_id
        break

# Find a numeric field by attempting to cast each column
numeric_field_id = None
for col in df.columns:
    # Try to infer if numeric
    try:
        vals = pd.to_numeric(df[col], errors='coerce')
        # If sufficient number of non-NaNs, treat as numeric
        if vals.notna().sum() > 0:
            numeric_field_id = col
            break
    except:
        continue

if numeric_field_id is None:
    print('Could not find any numeric field.')
else:
    print(f"Using numeric field (by @id): {numeric_field_id}")
# Now filter on this field if found
if numeric_field_id is not None:
    # Convert column to numeric
    df_numeric = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = df_numeric.median() if not np.isnan(df_numeric.median()) else 0
    filtered_df = df[df_numeric > threshold].copy()
    print(f"Filtered records in {selected_rs_id} with {numeric_field_id} > {threshold} (median)")
    display(filtered_df.head())
    
    # Normalize the numeric field for filtered records
    filtered_df[f"{numeric_field_id}_normalized"] = (
        pd.to_numeric(filtered_df[numeric_field_id], errors='coerce') - df_numeric.mean()
    ) / df_numeric.std()
    print(f"Normalized values in {numeric_field_id} (z-score):")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    # Try to group by a categorical field (choose the first non-numeric)
    group_field_id = None
    for col in df.columns:
        if col == numeric_field_id:
            continue
        try:
            if pd.api.types.is_numeric_dtype(pd.to_numeric(df[col], errors='coerce')):
                continue
        except:
            pass
        group_field_id = col
        break
    if group_field_id is not None:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"Grouped mean of {numeric_field_id} by {group_field_id}:")
        display(grouped_df.head())
    else:
        print('No suitable grouping field found.')
else:
    print('No numeric field found for EDA.')

## 5. Visualization
Visualize the numeric field's distribution or the relationship between two fields using matplotlib or seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is not None:
    plt.figure(figsize=(7, 4))
    sns.histplot(pd.to_numeric(df[numeric_field_id], errors='coerce').dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # If we found a grouping field, show mean numeric field value by group
    if group_field_id is not None:
        group_means = df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        plt.figure(figsize=(8, 4))
        sns.barplot(data=group_means, x=group_field_id, y=numeric_field_id)
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print('No numeric field found for visualization.')

## 6. Conclusion

In this notebook, we loaded and explored the FAIR² clinical dataset using the `mlcroissant` library. We:

- Examined available record sets and their fields by `@id`.
- Loaded and previewed data using Croissant schema identifiers for best interoperability and reproducibility.
- Performed basic EDA and visualized the distribution of a numeric variable.

This approach enables robust, schema-driven tabular data workflows for biomedical and clinical studies.